# ⭐ Fantasy World Predictions — Star Wars Universe
## Step 2: Cleaning

---

### 🎯 Goal of this notebook
Fix every issue flagged in the EDA notebook. **Key difference from LOTR/HP:** with only 87
rows, we can't afford to drop rows the way we did before — every decision here favors keeping
rows over dropping them.

**Decisions:**
- **No rows dropped, anywhere.** `mass`/`birth_year` outliers (Jabba, Yoda) are kept — real
  values, and Random Forest (which we'll compare against logreg/linreg in the pipeline notebook)
  handles outliers far better than a linear model, so we lean on model choice instead of losing data.
- `sex`/`gender`: fill the 4 missing values each as `"unknown"` rather than dropping
- `height`, `mass`, `birth_year`: **left as true NaN** — they're both target *and* feature
  candidates for each other, so imputation happens at the pipeline stage, not here
- `species` → `species_group` (`Human`/`Droid`/`Other`/`Unknown`) — the raw column's 36 unique
  values across 87 rows made it unusable
- `homeworld` → `homeworld_group` (top 5 worlds + `Other`/`Unknown`) — same sparsity problem
- `hair_color`/`skin_color`/`eye_color`: normalize casing, keep first value when multiple listed,
  group rare values (<3 occurrences) into `other` — `unknown` kept as its own bucket, not grouped in
- `films` → `film_count`; `vehicles`/`starships` → `has_vehicle`/`has_starship` binary flags

---

## 📦 Section 1: Imports & Load

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/star_wars_character_dataset.csv')
print(f'Starting shape: {df.shape}')

## 🚻 Section 2: Fill `sex` / `gender` missing values

Only 4 missing each, out of 87 rows — filling as `"unknown"` rather than dropping preserves
~5% of an already tiny dataset.

In [ ]:
df['sex'] = df['sex'].fillna('unknown').str.strip().str.lower()
df['gender'] = df['gender'].fillna('unknown').str.strip().str.lower()

print(df['sex'].value_counts())
print()
print(df['gender'].value_counts())

## 🤖 Section 3: Engineer `species_group` from `species`

36 unique values across 87 rows was unusable — collapse to the only categories with real
sample size.

In [ ]:
def species_group(val):
    if pd.isna(val):
        return 'Unknown'
    val = str(val)
    return val if val in ('Human', 'Droid') else 'Other'

df['species_group'] = df['species'].apply(species_group)
df['species_group'].value_counts()

## 🌍 Section 4: Engineer `homeworld_group` from `homeworld`

Same sparsity fix — keep the handful of worlds with real counts, bucket the rest.

In [ ]:
top_worlds = ['Naboo', 'Tatooine', 'Alderaan', 'Kamino', 'Coruscant']

def homeworld_group(val):
    if pd.isna(val):
        return 'Unknown'
    return val if val in top_worlds else 'Other'

df['homeworld_group'] = df['homeworld'].apply(homeworld_group)
df['homeworld_group'].value_counts()

## 🎨 Section 5: Clean `hair_color`, `skin_color`, `eye_color`

Take the first value when multiple are listed, normalize casing, group rare values (fewer than
3 occurrences) into `other`. `unknown` is kept as its own bucket, not merged into `other`, since
it means something different (we genuinely don't know) than a real-but-rare color.

In [ ]:
def first_val(val):
    if pd.isna(val):
        return 'unknown'
    return str(val).split(',')[0].strip().lower()

for col in ['hair_color', 'skin_color', 'eye_color']:
    df[col] = df[col].apply(first_val)
    counts = df[col].value_counts()
    rare = counts[counts < 3].index
    df[col] = df[col].apply(lambda x: 'other' if (x in rare and x != 'unknown') else x)
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()

## 🎬 Section 6: Engineer `film_count`, `has_vehicle`, `has_starship`

`films` is comma-separated text → count. `vehicles`/`starships` are ~85-90% missing → binary
presence flags instead of the raw text.

In [ ]:
df['film_count'] = df['films'].apply(lambda x: len(str(x).split(',')))
df['has_vehicle'] = df['vehicles'].notna().astype(int)
df['has_starship'] = df['starships'].notna().astype(int)

df[['film_count', 'has_vehicle', 'has_starship']].describe()

## 🧹 Section 7: Drop raw / replaced columns

`species`, `homeworld`, `films`, `vehicles`, `starships` are dropped — replaced by their
engineered versions. `height`, `mass`, `birth_year` are deliberately **not** touched here — they
stay with their real NaNs. `name` is kept for display only.

In [ ]:
df_clean = df.drop(columns=['species', 'homeworld', 'films', 'vehicles', 'starships'])
df_clean.head()

## ✅ Section 8: Final check & save

In [ ]:
print('=== Final clean dataset ===')
print(f'Shape: {df_clean.shape}  (no rows dropped — started and ended at 87)')
print()
print('Dtypes:')
print(df_clean.dtypes)
print()
print('Remaining missing values (expected: only height/mass/birth_year):')
print(df_clean.isnull().sum())

In [ ]:
df_clean.to_csv('../data/sw_characters_clean.csv', index=False)
print('✅ Saved to sw_characters_clean.csv')

---
## 🔜 What's Next?

Next: **`SW_ML_Pipeline.ipynb`** — same shared-factory idea as LOTR/HP, but with two new
twists this dataset requires:
1. **Imputation step** in the preprocessing pipeline (median for numeric, most-frequent for
   categorical), since `height`/`mass`/`birth_year` still carry real NaNs and can appear as
   *features* even when they're not the current target.
2. **Regression targets** (`height`, `mass`) alongside the classification ones (`sex`, `gender`,
   `species_group`) — first time we need regression metrics (MAE, R²) instead of only
   classification metrics.

> **Tell your assistant:** *"SW cleaning looks good, let's move to SW pipeline."*
